# ⚡ Indian Smart Grid Electricity Load Forecasting: Weighted Hybrid Ensemble Model
### Academic PBL Project • 5th Semester • Department of Computer Engineering / Data Science

---

## 📌 Project Overview
Electrical load forecasting is a cornerstone of intelligent power grid management. Accurate forecasts of national electricity demand prevent blackouts, reduce transmission line congestion, optimize power dispatch, and integrate renewable energy resources efficiently into the Indian National Grid.

This notebook implements an advanced **Weighted Hybrid Ensemble Model** combining five diverse statistical, machine learning, and deep learning architectures:
1. **Linear Regression (LR)** — Classical linear baseline capturing macro trends.
2. **Support Vector Regression (SVR)** — Kernel-based regression capturing non-linear boundary margins.
3. **Random Forest Regression (RFR)** — Non-parametric bagging ensemble capturing non-linear feature interactions.
4. **XGBoost Regression (XGB)** — Optimized gradient boosted decision trees minimizing residual errors.
5. **Long Short-Term Memory (LSTM)** — Deep recurrent neural network modeling 24-hour temporal sequential dynamics.

---

## 🎯 Hybrid Ensemble Formula
The five models are combined using an exact, fixed-weight ensemble:

$$\text{Final Prediction} = 0.05 \times \text{LR} + 0.15 \times \text{SVR} + 0.25 \times \text{RF} + 0.30 \times \text{XGBoost} + 0.25 \times \text{LSTM}$$

$$\sum w_i = 0.05 + 0.15 + 0.25 + 0.30 + 0.25 = 1.00$$

* **Target Variable:** `National Hourly Demand`
* **Unit:** `MW` (Megawatts)
* **LSTM Temporal Sequence Length:** 24 Hours
* **Train / Test Split:** 80% Chronological Training / 20% Testing
* **Deployment Interface:** Modern Light-Themed Glassmorphism Gradio Dashboard


In [1]:
# Step 1.1: Dependency and Environment Verification
import sys
import os
import json
import warnings
from pathlib import Path

# Suppress minor deprecation warnings for clean presentation
warnings.filterwarnings("ignore")

# Core Data Science & Plotting Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set plotting aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

# Machine Learning Libraries
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Gradient Boosting
try:
    from xgboost import XGBRegressor
except ImportError:
    print("Installing XGBoost...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBRegressor

# Deep Learning (TensorFlow / Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Web Interface
import gradio as gr

print("=" * 60)
print("ALL REQUIRED LIBRARIES IMPORTED SUCCESSFULLY")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"Pandas Version    : {pd.__version__}")
print(f"NumPy Version     : {np.__version__}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Gradio Version    : {gr.__version__}")
print("=" * 60)


ALL REQUIRED LIBRARIES IMPORTED SUCCESSFULLY
Python Version    : 3.14.0
Pandas Version    : 3.0.2
NumPy Version     : 2.4.4
TensorFlow Version: 2.22.0-rc0
Gradio Version    : 6.26.0


---
# 🔍 Step 2: Automatically Find Dataset Path
To ensure complete portability between local environments, JupyterLab, and Google Colab (without relying on hard-coded paths like `/content/drive/...`), this notebook scans candidate files dynamically and selects the dataset containing `National Hourly Demand`.


In [2]:
# Step 2.1: Automated Dataset Discovery
possible_extensions = ["*.xlsx", "*.xls", "*.csv"]
candidate_files = []

# Search current directory, parent directory, and subdirectories
search_roots = [Path("."), Path("..")]

for root in search_roots:
    for extension in possible_extensions:
        try:
            candidate_files.extend(root.rglob(extension))
        except Exception:
            pass

# Remove duplicate paths
candidate_files = sorted(list({f.resolve() for f in candidate_files}))

print(f"Found {len(candidate_files)} candidate dataset files in workspace:")
for idx, f in enumerate(candidate_files, 1):
    print(f"  [{idx}] {f.name} ({f.stat().st_size / (1024*1024):.2f} MB) -> {f}")

# Target column to identify the correct Indian electricity dataset
target_column = "National Hourly Demand"
dataset_path = None

for candidate in candidate_files:
    try:
        # Check header without loading full file
        if str(candidate).lower().endswith(".csv"):
            preview_df = pd.read_csv(candidate, nrows=2)
        else:
            preview_df = pd.read_excel(candidate, nrows=2)
            
        if target_column in preview_df.columns:
            dataset_path = candidate
            break
    except Exception:
        continue

if dataset_path is None:
    raise FileNotFoundError(f"Could not locate dataset containing '{target_column}' in candidate files!")

print("\n" + "=" * 60)
print("Dataset found: YES")
print(f"Dataset path: {dataset_path}")
print(f"Filename    : {dataset_path.name}")
print("=" * 60)


Found 9 candidate dataset files in workspace:
  [1] household_power_consumption.csv (126.80 MB) -> D:\Documents (D)\PBL_Model\FRANCE_DATASET\household_power_consumption.csv
  [2] lstm_predictions.csv (0.18 MB) -> D:\Documents (D)\PBL_Model\FRANCE_DATASET\LSTM_Output\lstm_predictions.csv
  [3] random_forest_predictions.csv (0.11 MB) -> D:\Documents (D)\PBL_Model\FRANCE_DATASET\RFR_Outputs\random_forest_predictions.csv
  [4] svr_predictions.csv (0.11 MB) -> D:\Documents (D)\PBL_Model\FRANCE_DATASET\SVR_Output\svr_predictions.csv
  [5] xgboost_predictions.csv (0.07 MB) -> D:\Documents (D)\PBL_Model\FRANCE_DATASET\XGBoost_Output\xgboost_predictions.csv
  [6] hourlyLoadDataIndia.xlsx (3.15 MB) -> D:\Documents (D)\PBL_Model\INDIAN_DATASET\dataset\hourlyLoadDataIndia.xlsx
  [7] monthly_temp.xlsx (0.01 MB) -> D:\Documents (D)\PBL_Model\INDIAN_DATASET\dataset\monthly_temp.xlsx
  [8] Hybrid_Model_Metrics.csv (0.00 MB) -> D:\Documents (D)\PBL_Model\INDIAN_DATASET\Hybrid_Model_Metrics.csv
  [9] Hy

---
# 📊 Step 3: Load and Verify Dataset
We load the detected dataset using pandas, convert the `datetime` column safely to standard datetime objects, remove any invalid timestamps, and sort the time series chronologically.


In [3]:
# Step 3.1: Load Dataset into Pandas DataFrame
print(f"Loading dataset from: {dataset_path} ...")
if str(dataset_path).lower().endswith(".csv"):
    df = pd.read_csv(dataset_path)
else:
    df = pd.read_excel(dataset_path)

print("\n--- Dataset Dimensions ---")
print(f"Number of rows   : {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")

print("\n--- First 5 Records ---")
display(df.head())

print("\n--- Missing Values Summary ---")
print(df.isnull().sum())

# Step 3.2: Clean and Chronologically Sort Datetime
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
initial_len = len(df)
df = df.dropna(subset=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

if len(df) < initial_len:
    print(f"Removed {initial_len - len(df)} records with invalid datetimes.")

print("\n--- Temporal Range ---")
print(f"Start Timestamp: {df['datetime'].min()}")
print(f"End Timestamp  : {df['datetime'].max()}")
print(f"Total Hours    : {len(df):,}")
print(f"Target Column  : {target_column} (MW)")


Loading dataset from: D:\Documents (D)\PBL_Model\INDIAN_DATASET\dataset\hourlyLoadDataIndia.xlsx ...

--- Dataset Dimensions ---
Number of rows   : 46,728
Number of columns: 7
Columns: ['datetime', 'National Hourly Demand', 'Northen Region Hourly Demand', 'Western Region Hourly Demand', 'Eastern Region Hourly Demand', 'Southern Region Hourly Demand', 'North-Eastern Region Hourly Demand']

--- First 5 Records ---
             datetime  ...  North-Eastern Region Hourly Demand
0 2019-01-01 00:00:00  ...                             1665.72
1 2019-01-01 01:00:00  ...                             1556.24
2 2019-01-01 02:00:00  ...                             1485.44
3 2019-01-01 03:00:00  ...                             1454.96
4 2019-01-01 04:00:00  ...                             1463.14

[5 rows x 7 columns]

--- Missing Values Summary ---
datetime                              0
National Hourly Demand                0
Northen Region Hourly Demand          0
Western Region Hourly Demand    

---
# 📈 Step 4: Exploratory Data Analysis & Target Overview
We inspect the statistical properties of the national demand in Megawatts (MW) and plot the full historical series.


In [4]:
# Step 4.1: Target Variable Summary Statistics
print("Summary Statistics for National Hourly Demand (MW):")
display(df[[target_column]].describe().T)

# Step 4.2: Plot Historical Electricity Demand
plt.figure(figsize=(16, 5), dpi=100)
plt.plot(df['datetime'], df[target_column], color='#0284c7', linewidth=0.8, alpha=0.9, label='National Demand (MW)')
plt.title("Indian National Electricity Demand Telemetry (2019 - 2024)", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Datetime", fontsize=12)
plt.ylabel("Electricity Demand (MW)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


Summary Statistics for National Hourly Demand (MW):
                          count           mean  ...          75%        max
National Hourly Demand  46728.0  160487.065667  ...  176633.6075  237361.97

[1 rows x 8 columns]


---
# ⚙️ Step 5: Feature Engineering
To maintain strict consistency with the existing individual model implementations (`LR_Gradio.ipynb`, `SVR_Gradio.ipynb`, `RFR_Gradio.ipynb`, `XGBoost_Gradio.ipynb`), we engineer calendar and cyclical temporal features:
* `hour` (0–23)
* `day` (1–31)
* `day_of_week` (0=Monday, 6=Sunday)
* `month` (1–12)
* `year` (2019–2024)
* `day_of_year` (1–366)

We also demonstrate historical lag and rolling features (`lag_1`, `lag_24`, `lag_48`, `lag_168`, `rolling_mean_24`, `rolling_mean_168`) to verify temporal autocorrelation without data leakage.


In [5]:
# Step 5.1: Create Temporal Features
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year
df['day_of_year'] = df['datetime'].dt.dayofyear

# Canonical features matching existing project notebooks
features = [
    'hour',
    'day',
    'day_of_week',
    'month',
    'year',
    'day_of_year'
]

# Step 5.2: Create Lag & Rolling Features for Exploratory Analysis
df['lag_1'] = df[target_column].shift(1)
df['lag_24'] = df[target_column].shift(24)
df['lag_48'] = df[target_column].shift(48)
df['lag_168'] = df[target_column].shift(168)
df['rolling_mean_24'] = df[target_column].shift(1).rolling(window=24).mean()
df['rolling_mean_168'] = df[target_column].shift(1).rolling(window=168).mean()

print("Feature Engineering Completed:")
print(f"Features: {features}")
print(f"Target  : {target_column}")
display(df[['datetime', target_column] + features + ['lag_1', 'lag_24', 'rolling_mean_24']].head())


Feature Engineering Completed:
Features: ['hour', 'day', 'day_of_week', 'month', 'year', 'day_of_year']
Target  : National Hourly Demand
             datetime  National Hourly Demand  ...  lag_24  rolling_mean_24
0 2019-01-01 00:00:00               118690.67  ...     NaN              NaN
1 2019-01-01 01:00:00               116029.23  ...     NaN              NaN
2 2019-01-01 02:00:00               114044.14  ...     NaN              NaN
3 2019-01-01 03:00:00               113648.97  ...     NaN              NaN
4 2019-01-01 04:00:00               116290.05  ...     NaN              NaN

[5 rows x 11 columns]


---
# ✂️ Step 6: Chronological Train / Test Split
For time-series electricity forecasting, random shuffling causes severe data leakage. We apply a strict chronological split:
* **80% Training:** Older historical records
* **20% Testing:** Recent unseen future records


In [6]:
# Step 6.1: Chronological 80/20 Train/Test Split
X = df[features]
y = df[target_column]

split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("=" * 60)
print("TIME-SERIES CHRONOLOGICAL SPLIT SUMMARY")
print(f"Total Dataset Samples: {len(df):,}")
print(f"Training Samples (80%): {len(X_train):,}")
print(f"  Training Period    : {train_df['datetime'].min()}  to  {train_df['datetime'].max()}")
print(f"Testing Samples (20%) : {len(X_test):,}")
print(f"  Testing Period     : {test_df['datetime'].min()}  to  {test_df['datetime'].max()}")
print("=" * 60)


TIME-SERIES CHRONOLOGICAL SPLIT SUMMARY
Total Dataset Samples: 46,728
Training Samples (80%): 37,382
  Training Period    : 2019-01-01 00:00:00  to  2023-04-07 13:00:00
Testing Samples (20%) : 9,346
  Testing Period     : 2023-04-07 14:00:00  to  2024-04-30 23:00:00


---
# 📐 Step 7: Model 1 — Linear Regression (Weight: 0.05)
Linear Regression serves as the fundamental baseline model, estimating a linear relationship between temporal features and electricity load.


In [7]:
# Step 7.1: Train Linear Regression Model
print("Training Linear Regression Model...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Generate predictions for the test set
lr_prediction = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_prediction)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_prediction))
lr_r2 = r2_score(y_test, lr_prediction)

print("Linear Regression Model Trained Successfully!")
print(f"  MAE : {lr_mae:,.2f} MW")
print(f"  RMSE: {lr_rmse:,.2f} MW")
print(f"  R²  : {lr_r2:.4f}")


Training Linear Regression Model...
Linear Regression Model Trained Successfully!
  MAE : 17,303.20 MW
  RMSE: 20,854.42 MW
  R²  : -0.1411


---
# 🛡️ Step 8: Model 2 — Support Vector Regression (SVR) (Weight: 0.15)
Support Vector Regression with a Radial Basis Function (RBF) kernel maps features into a high-dimensional space. Features are standardized using `StandardScaler`.
If a pre-trained SVR model and scaler are saved in the project (`SVR Model Data/`), we load them to preserve the exact project configuration and prevent slow RBF retraining.


In [8]:
# Step 8.1: Support Vector Regression Setup & Scaling
# Search for saved SVR model artifacts
svr_model_candidates = list(Path(".").rglob("svr_model.pkl"))
svr_scaler_candidates = list(Path(".").rglob("svr_scaler.pkl"))

if svr_model_candidates and svr_scaler_candidates:
    print(f"Loading pre-trained SVR model from : {svr_model_candidates[0]}")
    print(f"Loading pre-trained SVR scaler from: {svr_scaler_candidates[0]}")
    svr_model = joblib.load(svr_model_candidates[0])
    svr_scaler = joblib.load(svr_scaler_candidates[0])
    X_test_scaled = svr_scaler.transform(X_test)
    print("Pre-trained SVR Model loaded successfully!")
else:
    print("Training SVR from scratch using project hyperparameters...")
    svr_scaler = StandardScaler()
    X_train_scaled = svr_scaler.fit_transform(X_train)
    X_test_scaled = svr_scaler.transform(X_test)
    
    svr_model = SVR(
        kernel='rbf',
        C=100,
        gamma='scale',
        epsilon=0.1
    )
    svr_model.fit(X_train_scaled, y_train)
    print("SVR Model trained successfully!")

# Generate SVR test set predictions
svr_prediction = svr_model.predict(X_test_scaled)

svr_mae = mean_absolute_error(y_test, svr_prediction)
svr_rmse = np.sqrt(mean_squared_error(y_test, svr_prediction))
svr_r2 = r2_score(y_test, svr_prediction)

print(f"SVR Predictions Generated:")
print(f"  MAE : {svr_mae:,.2f} MW")
print(f"  RMSE: {svr_rmse:,.2f} MW")
print(f"  R²  : {svr_r2:.4f}")


Loading pre-trained SVR model from : SVR Model Data\svr_model.pkl
Loading pre-trained SVR scaler from: SVR Model Data\svr_scaler.pkl
Pre-trained SVR Model loaded successfully!
SVR Predictions Generated:
  MAE : 16,883.06 MW
  RMSE: 20,446.05 MW
  R²  : -0.0968


---
# 🌲 Step 9: Model 3 — Random Forest Regression (Weight: 0.25)
Random Forest Regression creates an ensemble of 100 decision trees to capture non-linear relationships and interactions among calendar features.


In [9]:
# Step 9.1: Train Random Forest Regressor
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Generate predictions for the test set
rf_prediction = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_prediction)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_prediction))
rf_r2 = r2_score(y_test, rf_prediction)

print("Random Forest Regressor Trained Successfully!")
print(f"  MAE : {rf_mae:,.2f} MW")
print(f"  RMSE: {rf_rmse:,.2f} MW")
print(f"  R²  : {rf_r2:.4f}")


Training Random Forest Regressor...
Random Forest Regressor Trained Successfully!
  MAE : 16,030.64 MW
  RMSE: 19,612.99 MW
  R²  : -0.0093


---
# 🚀 Step 10: Model 4 — XGBoost Regression (Weight: 0.30)
XGBoost (Extreme Gradient Boosting) iteratively fits regression trees to residual errors, optimizing a squared-error loss function with regularization.


In [10]:
# Step 10.1: Train XGBoost Regressor
print("Training XGBoost Regressor...")
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Generate predictions for the test set
xgb_prediction = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_prediction)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_prediction))
xgb_r2 = r2_score(y_test, xgb_prediction)

print("XGBoost Regressor Trained Successfully!")
print(f"  MAE : {xgb_mae:,.2f} MW")
print(f"  RMSE: {xgb_rmse:,.2f} MW")
print(f"  R²  : {xgb_r2:.4f}")


Training XGBoost Regressor...
XGBoost Regressor Trained Successfully!
  MAE : 16,304.42 MW
  RMSE: 19,745.77 MW
  R²  : -0.0230


---
# 🧠 Step 11: Model 5 — Long Short-Term Memory (LSTM) (Weight: 0.25)
LSTM is a deep recurrent neural network specialized in capturing long-term dependencies across hourly electrical sequences.
* **Sequence Length (`TIME_STEPS`):** 24 hours (1 full daily cycle)
* **Architecture:** 2 LSTM layers (64 & 32 units) with 20% Dropout, Dense(16, relu), Dense(1)
* **Scaling:** `MinMaxScaler(0, 1)` fitted exclusively on training data
* **Alignment:** By concatenating `train_scaled[-TIME_STEPS:]` with `test_scaled`, test sequence $i$ accurately forecasts test timestamp $i$.


In [11]:
# Step 11.1: Prepare LSTM Sequences with 24-Hour Lookback
TIME_STEPS = 24

# Scale target using MinMaxScaler
lstm_scaler = MinMaxScaler(feature_range=(0, 1))

train_target_vals = train_df[[target_column]].values
test_target_vals = test_df[[target_column]].values

train_scaled = lstm_scaler.fit_transform(train_target_vals)
test_scaled = lstm_scaler.transform(test_target_vals)

def create_sequences(values, time_steps):
    X_seq, y_seq = [], []
    for i in range(time_steps, len(values)):
        X_seq.append(values[i-time_steps:i, 0])
        y_seq.append(values[i, 0])
    return np.array(X_seq), np.array(y_seq)

# Training sequences
X_train_lstm, y_train_lstm = create_sequences(train_scaled, TIME_STEPS)
X_train_lstm = X_train_lstm.reshape((X_train_lstm.shape[0], X_train_lstm.shape[1], 1))

# Testing sequences: include last TIME_STEPS from train so test output matches len(test_df)
combined_test_input = np.concatenate([train_scaled[-TIME_STEPS:], test_scaled])
X_test_lstm, _ = create_sequences(combined_test_input, TIME_STEPS)
X_test_lstm = X_test_lstm.reshape((X_test_lstm.shape[0], X_test_lstm.shape[1], 1))

print(f"LSTM Training Sequences Shape: {X_train_lstm.shape}")
print(f"LSTM Testing Sequences Shape : {X_test_lstm.shape} (Matches test_df: {len(X_test_lstm) == len(test_df)})")

# Step 11.2: Build and Compile LSTM Architecture or Load Existing
lstm_model_dir = Path("LSTM Model Data")
lstm_model_path = lstm_model_dir / "lstm_model.keras"

if lstm_model_path.exists():
    print(f"Loading pre-trained LSTM model from: {lstm_model_path}")
    lstm_model = tf.keras.models.load_model(lstm_model_path)
else:
    lstm_model = Sequential([
        Input(shape=(TIME_STEPS, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])

    lstm_model.compile(optimizer='adam', loss='mean_squared_error')
    lstm_model.summary()

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True
    )

    print("\nTraining LSTM Model...")
    history = lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=12,
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=1
    )
    
    lstm_model_dir.mkdir(parents=True, exist_ok=True)
    lstm_model.save(lstm_model_path)
    print(f"LSTM Model saved to {lstm_model_path}")

# Generate test predictions
lstm_pred_scaled = lstm_model.predict(X_test_lstm, verbose=0)
lstm_prediction = lstm_scaler.inverse_transform(lstm_pred_scaled).flatten()

lstm_mae = mean_absolute_error(y_test, lstm_prediction)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_prediction))
lstm_r2 = r2_score(y_test, lstm_prediction)

print("\nLSTM Model Trained Successfully!")
print(f"  MAE : {lstm_mae:,.2f} MW")
print(f"  RMSE: {lstm_rmse:,.2f} MW")
print(f"  R²  : {lstm_r2:.4f}")


LSTM Training Sequences Shape: (37358, 24, 1)
LSTM Testing Sequences Shape : (9346, 24, 1) (Matches test_df: True)
Model: "sequential"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)   

---
# 🔗 Step 12: Align All Five Model Predictions
Before computing the hybrid ensemble forecast, we align predictions from all five models with the actual test demand values into a unified prediction DataFrame.


In [12]:
# Step 12.1: Construct Unified Prediction DataFrame
print("Verifying prediction array dimensions:")
print(f"  Actual values length: {len(y_test)}")
print(f"  LR predictions      : {len(lr_prediction)}")
print(f"  SVR predictions     : {len(svr_prediction)}")
print(f"  RF predictions      : {len(rf_prediction)}")
print(f"  XGB predictions     : {len(xgb_prediction)}")
print(f"  LSTM predictions    : {len(lstm_prediction)}")

assert len(y_test) == len(lr_prediction) == len(svr_prediction) == len(rf_prediction) == len(xgb_prediction) == len(lstm_prediction), "Prediction lengths do not match!"

prediction_df = pd.DataFrame({
    "datetime": test_df["datetime"].values,
    "Actual": y_test.values,
    "Linear_Regression": lr_prediction.flatten(),
    "SVR": svr_prediction.flatten(),
    "Random_Forest": rf_prediction.flatten(),
    "XGBoost": xgb_prediction.flatten(),
    "LSTM": lstm_prediction.flatten()
})

print(f"\nUnified Prediction DataFrame Constructed: {prediction_df.shape}")
display(prediction_df.head(10))


Verifying prediction array dimensions:
  Actual values length: 9346
  LR predictions      : 9346
  SVR predictions     : 9346
  RF predictions      : 9346
  XGB predictions     : 9346
  LSTM predictions    : 9346

Unified Prediction DataFrame Constructed: (9346, 7)
             datetime     Actual  ...        XGBoost           LSTM
0 2023-04-07 14:00:00  183298.80  ...  187451.656250  177786.281250
1 2023-04-07 15:00:00  183642.74  ...  187399.890625  176843.703125
2 2023-04-07 16:00:00  180148.32  ...  183894.703125  177781.375000
3 2023-04-07 17:00:00  173698.41  ...  176186.468750  176452.609375
4 2023-04-07 18:00:00  174092.89  ...  175474.031250  172761.500000
5 2023-04-07 19:00:00  182541.83  ...  181876.000000  173351.500000
6 2023-04-07 20:00:00  178704.10  ...  178169.328125  179531.828125
7 2023-04-07 21:00:00  175628.75  ...  175983.656250  178028.125000
8 2023-04-07 22:00:00  175838.78  ...  176014.359375  174215.921875
9 2023-04-07 23:00:00  174420.57  ...  174775.468750  

---
# ⚖️ Step 13: Calculate Weighted Hybrid Ensemble Prediction
We apply the exact hybrid formula and verify programmatically that the weights sum to 1.0:

```python
final_prediction = (
    0.05 * prediction_df["Linear_Regression"] +
    0.15 * prediction_df["SVR"] +
    0.25 * prediction_df["Random_Forest"] +
    0.30 * prediction_df["XGBoost"] +
    0.25 * prediction_df["LSTM"]
)
```


In [13]:
# Step 13.1: Verify Weights & Compute Hybrid Ensemble Forecast
weights = {
    "Linear Regression": 0.05,
    "SVR": 0.15,
    "Random Forest": 0.25,
    "XGBoost": 0.30,
    "LSTM": 0.25
}

weight_sum = sum(weights.values())
print("=" * 50)
print(f"Weight Sum: {weight_sum:.4f}")
for model_name, w in weights.items():
    print(f"  • {model_name:<18}: {w:.2f} ({int(w*100)}%)")
print("=" * 50)

assert np.isclose(weight_sum, 1.0), "Error: Model weights must sum exactly to 1.0!"

# Compute Hybrid Ensemble Prediction
final_prediction = (
    0.05 * prediction_df["Linear_Regression"] +
    0.15 * prediction_df["SVR"] +
    0.25 * prediction_df["Random_Forest"] +
    0.30 * prediction_df["XGBoost"] +
    0.25 * prediction_df["LSTM"]
)

prediction_df["Hybrid_Ensemble"] = final_prediction

print("Hybrid Ensemble Forecast Generated Successfully!")
display(prediction_df[["datetime", "Actual", "Hybrid_Ensemble"]].head(10))


Weight Sum: 1.0000
  • Linear Regression : 0.05 (5%)
  • SVR               : 0.15 (15%)
  • Random Forest     : 0.25 (25%)
  • XGBoost           : 0.30 (30%)
  • LSTM              : 0.25 (25%)
Hybrid Ensemble Forecast Generated Successfully!
             datetime     Actual  Hybrid_Ensemble
0 2023-04-07 14:00:00  183298.80    184450.954567
1 2023-04-07 15:00:00  183642.74    184146.394358
2 2023-04-07 16:00:00  180148.32    181977.517482
3 2023-04-07 17:00:00  173698.41    177274.646352
4 2023-04-07 18:00:00  174092.89    175640.722119
5 2023-04-07 19:00:00  182541.83    179093.386489
6 2023-04-07 20:00:00  178704.10    178359.406897
7 2023-04-07 21:00:00  175628.75    176437.893788
8 2023-04-07 22:00:00  175838.78    175009.370041
9 2023-04-07 23:00:00  174420.57    173498.548042


---
# 📏 Step 14 & 15: Model Evaluation, Comparison & Best Model Determination
We evaluate all five individual models and the Hybrid Ensemble using standard regression metrics:
* **MAE** (Mean Absolute Error, in MW)
* **MSE** (Mean Squared Error, in $\text{MW}^2$)
* **RMSE** (Root Mean Squared Error, in MW)
* **$R^2$** (Coefficient of Determination)


In [14]:
# Step 14.1: Compute Metrics for All Models
model_columns = [
    ("Linear Regression", "Linear_Regression"),
    ("SVR", "SVR"),
    ("Random Forest", "Random_Forest"),
    ("XGBoost", "XGBoost"),
    ("LSTM", "LSTM"),
    ("Hybrid Ensemble", "Hybrid_Ensemble")
]

metrics_list = []

for display_name, col_name in model_columns:
    actual = prediction_df["Actual"]
    pred = prediction_df[col_name]
    
    mae = mean_absolute_error(actual, pred)
    mse = mean_squared_error(actual, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, pred)
    
    metrics_list.append({
        "Model": display_name,
        "MAE (MW)": round(mae, 2),
        "MSE (MW²)": round(mse, 2),
        "RMSE (MW)": round(rmse, 2),
        "R²": round(r2, 4)
    })

comparison_df = pd.DataFrame(metrics_list)

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON TABLE")
print("=" * 70)
display(comparison_df)

# Step 15.1: Automatically Determine Best Models
best_rmse_model = comparison_df.loc[comparison_df["RMSE (MW)"].idxmin(), "Model"]
best_mae_model = comparison_df.loc[comparison_df["MAE (MW)"].idxmin(), "Model"]
best_r2_model = comparison_df.loc[comparison_df["R²"].idxmax(), "Model"]

print("\n" + "=" * 50)
print("AUTOMATED PERFORMANCE LEADERS")
print(f"  Best Model based on Lowest RMSE: {best_rmse_model} ({comparison_df['RMSE (MW)'].min():,.2f} MW)")
print(f"  Best Model based on Lowest MAE : {best_mae_model} ({comparison_df['MAE (MW)'].min():,.2f} MW)")
print(f"  Best Model based on Highest R² : {best_r2_model} ({comparison_df['R²'].max():.4f})")
print("=" * 50)


MODEL PERFORMANCE COMPARISON TABLE
               Model  MAE (MW)     MSE (MW²)  RMSE (MW)      R²
0  Linear Regression  17303.20  4.349070e+08   20854.42 -0.1411
1                SVR  16883.06  4.180410e+08   20446.05 -0.0968
2      Random Forest  16030.64  3.846694e+08   19612.99 -0.0093
3            XGBoost  16304.42  3.898953e+08   19745.77 -0.0230
4               LSTM   4234.71  2.798995e+07    5290.55  0.9266
5    Hybrid Ensemble  12680.20  2.320222e+08   15232.28  0.3912

AUTOMATED PERFORMANCE LEADERS
  Best Model based on Lowest RMSE: LSTM (5,290.55 MW)
  Best Model based on Lowest MAE : LSTM (4,234.71 MW)
  Best Model based on Highest R² : LSTM (0.9266)


---
# 📊 Step 16: Visualizations
We generate clean, publication-ready figures for the PBL project:
1. **Plot 1:** Actual Demand vs Hybrid Predicted Demand.
2. **Plot 2:** Model Comparison Bar Chart (RMSE in MW).
3. **Plot 3:** Actual vs All Six Model Predictions (Zoomed view).
4. **Plot 4:** Prediction Residuals Distribution.


In [15]:
# Ensure output directory exists
output_dir = Path("Hybrid model data")
if not output_dir.exists() and (Path("INDIAN_DATASET") / "Hybrid model data").exists():
    output_dir = Path("INDIAN_DATASET") / "Hybrid model data"
output_dir.mkdir(parents=True, exist_ok=True)

# Step 16.1: Plot 1 — Actual vs Hybrid Predicted Electricity Demand
plt.figure(figsize=(16, 6), dpi=100)
# Plot slice of 300 test hours for clear readability
sample_slice = slice(0, 300)
plt.plot(prediction_df['datetime'].iloc[sample_slice], prediction_df['Actual'].iloc[sample_slice], 
         label='Actual Demand', color='#0f172a', linewidth=2.0)
plt.plot(prediction_df['datetime'].iloc[sample_slice], prediction_df['Hybrid_Ensemble'].iloc[sample_slice], 
         label='Hybrid Ensemble Forecast', color='#0284c7', linewidth=2.0, linestyle='--')

plt.title("Plot 1: Actual Demand vs Hybrid Ensemble Forecast (Test Slice)", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Datetime", fontsize=12)
plt.ylabel("Electricity Demand (MW)", fontsize=12)
plt.legend(frameon=True, facecolor='white', loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(output_dir / 'Plot_1_Actual_vs_Hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 16.2: Plot 2 — Model Comparison: RMSE (MW)
plt.figure(figsize=(12, 5), dpi=100)
colors = ['#94a3b8', '#64748b', '#38bdf8', '#818cf8', '#10b981', '#0284c7']
bars = plt.bar(comparison_df['Model'], comparison_df['RMSE (MW)'], color=colors, edgecolor='#0f172a', linewidth=1.2, width=0.55)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 350, f"{yval:,.1f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.title("Plot 2: Model RMSE Comparison (Megawatts - MW)", fontsize=14, fontweight='bold', pad=12)
plt.ylabel("RMSE (MW) - Lower is Better", fontsize=12)
plt.ylim(0, comparison_df['RMSE (MW)'].max() * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(output_dir / 'Plot_1_Actual_vs_Hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 16.3: Plot 3 — Actual Demand vs All Five Models and Hybrid Ensemble
plt.figure(figsize=(16, 7), dpi=100)
zoom_slice = slice(100, 250)
time_zoom = prediction_df['datetime'].iloc[zoom_slice]

plt.plot(time_zoom, prediction_df['Actual'].iloc[zoom_slice], label='Actual Demand', color='#0f172a', linewidth=2.5)
plt.plot(time_zoom, prediction_df['Linear_Regression'].iloc[zoom_slice], label='Linear Regression (5%)', color='#94a3b8', alpha=0.7, linestyle=':')
plt.plot(time_zoom, prediction_df['SVR'].iloc[zoom_slice], label='SVR (15%)', color='#c084fc', alpha=0.7, linestyle=':')
plt.plot(time_zoom, prediction_df['Random_Forest'].iloc[zoom_slice], label='Random Forest (25%)', color='#38bdf8', alpha=0.8, linestyle='-.')
plt.plot(time_zoom, prediction_df['XGBoost'].iloc[zoom_slice], label='XGBoost (30%)', color='#f97316', alpha=0.8, linestyle='-.')
plt.plot(time_zoom, prediction_df['LSTM'].iloc[zoom_slice], label='LSTM (25%)', color='#10b981', alpha=0.9, linewidth=1.8)
plt.plot(time_zoom, prediction_df['Hybrid_Ensemble'].iloc[zoom_slice], label='Hybrid Ensemble (100%)', color='#dc2626', linewidth=2.2)

plt.title("Plot 3: Multi-Model Load Forecast Trajectories vs Actual Demand", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Datetime", fontsize=12)
plt.ylabel("Electricity Demand (MW)", fontsize=12)
plt.legend(frameon=True, facecolor='white', loc='upper right', ncol=2)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig(output_dir / 'Plot_1_Actual_vs_Hybrid.png', dpi=150, bbox_inches='tight')
plt.show()


---
# 💾 Step 17: Save Prediction Results & Model Metadata
We export all final prediction time-series, comparative performance metrics, and model architecture documentation directly into the `Hybrid model data` directory.


In [16]:
# Step 17: Save All Output Files into 'Hybrid model data' Folder
output_dir = Path("Hybrid model data")
if not output_dir.exists() and (Path("INDIAN_DATASET") / "Hybrid model data").exists():
    output_dir = Path("INDIAN_DATASET") / "Hybrid model data"
output_dir.mkdir(parents=True, exist_ok=True)

# Step 17.1: Save Predictions CSV
predictions_csv_path = output_dir / "Hybrid_Model_Predictions.csv"
prediction_df.to_csv(predictions_csv_path, index=False)
print(f"Predictions saved to: {predictions_csv_path.resolve()} ({predictions_csv_path.stat().st_size / 1024:.1f} KB)")

# Step 17.2: Save Metrics CSV
metrics_csv_path = output_dir / "Hybrid_Model_Metrics.csv"
comparison_df.to_csv(metrics_csv_path, index=False)
print(f"Evaluation metrics saved to: {metrics_csv_path.resolve()}")

# Step 17.3: Save Hybrid Model Information Metadata
model_info = {
    "Model": "Weighted Hybrid Ensemble",
    "Weights": {
        "Linear Regression": 0.05,
        "SVR": 0.15,
        "Random Forest": 0.25,
        "XGBoost": 0.30,
        "LSTM": 0.25
    },
    "Formula": "final_prediction = 0.05*LR + 0.15*SVR + 0.25*RF + 0.30*XGB + 0.25*LSTM",
    "Target": target_column,
    "Unit": "MW",
    "LSTM_Sequence_Length_Hours": TIME_STEPS,
    "Train_Test_Split": "80/20 Chronological",
    "Performance": comparison_df.to_dict(orient="records")
}

info_json_path = output_dir / "Hybrid_Model_Info.json"
with open(info_json_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=4)
print(f"Model information JSON saved to: {info_json_path.resolve()}")

# Also save plain text specification
info_txt_path = output_dir / "Hybrid_Model_Info.txt"
with open(info_txt_path, "w", encoding="utf-8") as f:
    f.write("""Model: Weighted Hybrid Ensemble

Linear Regression Weight: 0.05
SVR Weight: 0.15
Random Forest Weight: 0.25
XGBoost Weight: 0.30
LSTM Weight: 0.25

Target: National Hourly Demand
Unit: MW
Sequence Length: 24 hours
Train/Test Split: 80/20 Chronological
""")
print(f"Model information TXT saved to: {info_txt_path.resolve()}")


Predictions saved to: D:\Documents (D)\PBL_Model\INDIAN_DATASET\Hybrid model data\Hybrid_Model_Predictions.csv (1141.3 KB)
Evaluation metrics saved to: D:\Documents (D)\PBL_Model\INDIAN_DATASET\Hybrid model data\Hybrid_Model_Metrics.csv
Model information JSON saved to: D:\Documents (D)\PBL_Model\INDIAN_DATASET\Hybrid model data\Hybrid_Model_Info.json
Model information TXT saved to: D:\Documents (D)\PBL_Model\INDIAN_DATASET\Hybrid model data\Hybrid_Model_Info.txt


---
# 🌐 Step 18: Modern Light-Themed Glassmorphism Gradio Interface
We build and launch an interactive web telemetry dashboard for the Hybrid Model.
* **Theme:** Clean, modern light palette (white, light cyan, sky blue, emerald green, soft violet).
* **Inputs:** Date (`YYYY-MM-DD`) and Time (`HH:00`).
* **Real-time Inference:** Directly executes the in-memory trained models (`lr_model`, `svr_model`, `rf_model`, `xgb_model`, `lstm_model`) without retraining.
* **Output:** Clear display of `Predicted National Electricity Demand: XXXX MW`, model weights, and breakdown of all component models with error handling.


In [17]:
# Step 18.1: Prediction Function with In-Memory Models & Robust Error Handling
def predict_hybrid_load(date, time_str):
    """
    End-to-end hybrid forecasting pipeline:
    Date + Time -> Feature Generation -> LR, SVR, RF, XGB, LSTM -> Weighted Ensemble -> Final MW
    """
    if not date or not str(date).strip():
        return "Error: Please enter a valid Date (e.g. 2024-04-01)."
    if not time_str or not str(time_str).strip():
        return "Error: Please enter a valid Time (e.g. 18:00)."

    try:
        # Parse timestamp safely
        datetime_value = pd.to_datetime(f"{date} {time_str}")
    except Exception as e:
        return f"Error: Invalid datetime format '{date} {time_str}'. Please use YYYY-MM-DD for Date and HH:00 for Time."

    try:
        # 1. Feature Generation for Tabular Models
        input_data = pd.DataFrame({
            'hour': [datetime_value.hour],
            'day': [datetime_value.day],
            'day_of_week': [datetime_value.dayofweek],
            'month': [datetime_value.month],
            'year': [datetime_value.year],
            'day_of_year': [datetime_value.dayofyear]
        })

        # 2. Linear Regression Prediction
        lr_p = float(lr_model.predict(input_data)[0])

        # 3. SVR Prediction (with scaling)
        input_scaled = svr_scaler.transform(input_data)
        svr_p = float(svr_model.predict(input_scaled)[0])

        # 4. Random Forest Prediction
        rf_p = float(rf_model.predict(input_data)[0])

        # 5. XGBoost Prediction
        xgb_p = float(xgb_model.predict(input_data)[0])

        # 6. LSTM Prediction (using 24-hour historical window)
        time_diffs = abs(df['datetime'] - datetime_value)
        nearest_idx = time_diffs.idxmin()
        pos = df.index.get_loc(nearest_idx)

        # Retrieve prior 24 hours of demand
        if pos >= TIME_STEPS:
            seq_values = df.iloc[pos - TIME_STEPS : pos][target_column].values.reshape(-1, 1)
        else:
            # Fallback to first available 24 hours if at the very beginning
            seq_values = df.iloc[:TIME_STEPS][target_column].values.reshape(-1, 1)

        seq_scaled = lstm_scaler.transform(seq_values)
        seq_input = seq_scaled.reshape(1, TIME_STEPS, 1)
        lstm_p_scaled = lstm_model.predict(seq_input, verbose=0)
        lstm_p = float(lstm_scaler.inverse_transform(lstm_p_scaled)[0][0])

        # 7. Exact Weighted Hybrid Ensemble Formula
        hybrid_p = (
            0.05 * lr_p +
            0.15 * svr_p +
            0.25 * rf_p +
            0.30 * xgb_p +
            0.25 * lstm_p
        )

        # Telemetry context if exact or near timestamp exists
        nearest_time = df.iloc[pos]['datetime']
        nearest_actual = df.iloc[pos][target_column]
        time_diff_hours = abs((datetime_value - nearest_time).total_seconds()) / 3600.0

        telemetry_info = ""
        if time_diff_hours <= 1.0:
            telemetry_info = f"\n• Dataset Timestamp Reference: {nearest_time}\n• Actual Recorded Demand     : {nearest_actual:,.2f} MW"

        response = (
            f"Predicted National Electricity Demand: {hybrid_p:,.2f} MW\n\n"
            f"Model: Hybrid Ensemble\n"
            f"Weights:\n"
            f"LR 5% | SVR 15% | RF 25% | XGBoost 30% | LSTM 25%\n\n"
            f"Component Predictions Breakdown:\n"
            f"• Linear Regression (05%): {lr_p:,.2f} MW\n"
            f"• SVR               (15%): {svr_p:,.2f} MW\n"
            f"• Random Forest     (25%): {rf_p:,.2f} MW\n"
            f"• XGBoost           (30%): {xgb_p:,.2f} MW\n"
            f"• LSTM              (25%): {lstm_p:,.2f} MW"
            f"{telemetry_info}"
        )
        return response

    except Exception as e:
        return f"Error: Prediction Failure: {str(e)}"

# Step 18.2: Test Prediction Function
sample_test = predict_hybrid_load("2024-04-01", "18:00")
print("Sample Prediction Test:")
print(sample_test)


Sample Prediction Test:
Predicted National Electricity Demand: 173,704.77 MW

Model: Hybrid Ensemble
Weights:
LR 5% | SVR 15% | RF 25% | XGBoost 30% | LSTM 25%

Component Predictions Breakdown:
• Linear Regression (05%): 183,653.56 MW
• SVR               (15%): 170,799.34 MW
• Random Forest     (25%): 169,601.25 MW
• XGBoost           (30%): 167,647.08 MW
• LSTM              (25%): 184,831.02 MW
• Dataset Timestamp Reference: 2024-04-01 18:00:00
• Actual Recorded Demand     : 185,414.67 MW


In [18]:
# Step 18.3: Modern Light-Themed Glassmorphism Gradio Interface
LIGHT_THEME_CSS = """
/* ==============================================================================
   CLEAN LIGHT-THEME GLASSMORPHISM SMART GRID TELEMETRY DASHBOARD CSS
   ============================================================================== */

:root, html, body, .gradio-container {
    color-scheme: light !important;
    background-color: #f8fafc !important;
    color: #0f172a !important;
    --body-background-fill: #f8fafc !important;
    --background-fill-primary: #ffffff !important;
    --background-fill-secondary: #f1f5f9 !important;
    --border-color-primary: rgba(148, 163, 184, 0.25) !important;
    --border-color-secondary: rgba(148, 163, 184, 0.15) !important;
    --block-background-fill: #ffffff !important;
    --block-border-color: rgba(226, 232, 240, 0.9) !important;
    --block-label-text-color: #475569 !important;
    --block-title-text-color: #0f172a !important;
    --input-background-fill: #ffffff !important;
    --input-border-color: rgba(203, 213, 225, 0.9) !important;
    --input-placeholder-color: #94a3b8 !important;
    --body-text-color: #0f172a !important;
    --body-text-color-subdued: #475569 !important;
}

body, html {
    background-color: #f8fafc !important;
    color: #0f172a !important;
}

.gradio-container {
    max-width: 1060px !important;
    margin: 0 auto !important;
    background: radial-gradient(circle at 10% 10%, #f0f9ff 0%, #f8fafc 60%, #faf5ff 100%) !important;
    color: #0f172a !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", sans-serif !important;
    min-height: 100vh !important;
}

@keyframes fadeInScale {
    0% { opacity: 0; transform: translateY(8px) scale(0.99); }
    100% { opacity: 1; transform: translateY(0) scale(1); }
}

/* Glass Header */
.sg-header-light {
    background: rgba(255, 255, 255, 0.85) !important;
    backdrop-filter: blur(20px) !important;
    -webkit-backdrop-filter: blur(20px) !important;
    border: 1px solid rgba(226, 232, 240, 0.8) !important;
    border-top: 3px solid #0284c7 !important;
    border-radius: 22px !important;
    padding: 28px 32px 24px !important;
    margin-bottom: 22px !important;
    text-align: center !important;
    box-shadow: 0 10px 30px -10px rgba(15, 23, 42, 0.08) !important;
    animation: fadeInScale 0.6s cubic-bezier(0.16, 1, 0.3, 1) !important;
    transition: all 0.3s ease !important;
}

.sg-header-light:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 16px 36px -10px rgba(15, 23, 42, 0.12), 0 0 20px rgba(2, 132, 199, 0.08) !important;
}

.sg-badge-light {
    display: inline-flex !important;
    align-items: center !important;
    gap: 8px !important;
    background: rgba(2, 132, 199, 0.08) !important;
    border: 1px solid rgba(2, 132, 199, 0.25) !important;
    color: #0284c7 !important;
    font-size: 11px !important;
    font-weight: 700 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.1em !important;
    padding: 5px 16px !important;
    border-radius: 9999px !important;
    margin-bottom: 12px !important;
}

.sg-title-light {
    font-size: 30px !important;
    font-weight: 800 !important;
    color: #0f172a !important;
    letter-spacing: -0.02em !important;
    margin: 0 0 4px 0 !important;
}

.sg-title-sub-light {
    font-size: 16px !important;
    font-weight: 600 !important;
    color: #0284c7 !important;
    margin: 0 0 10px 0 !important;
}

.sg-model-tag-light {
    display: inline-block !important;
    background: rgba(124, 58, 237, 0.08) !important;
    color: #7c3aed !important;
    font-size: 13px !important;
    font-weight: 700 !important;
    padding: 5px 16px !important;
    border-radius: 10px !important;
    border: 1px solid rgba(124, 58, 237, 0.25) !important;
    margin-bottom: 8px !important;
}

.sg-subtitle-light {
    font-size: 13px !important;
    color: #64748b !important;
    margin: 0 !important;
}

/* Glass Cards */
.sg-card-light {
    background: rgba(255, 255, 255, 0.85) !important;
    backdrop-filter: blur(16px) !important;
    -webkit-backdrop-filter: blur(16px) !important;
    border: 1px solid rgba(226, 232, 240, 0.85) !important;
    border-radius: 18px !important;
    padding: 18px 20px !important;
    margin-bottom: 16px !important;
    box-shadow: 0 6px 20px -5px rgba(15, 23, 42, 0.05) !important;
    transition: all 0.3s cubic-bezier(0.16, 1, 0.3, 1) !important;
    animation: fadeInScale 0.7s cubic-bezier(0.16, 1, 0.3, 1) !important;
}

.sg-card-light:hover {
    transform: translateY(-2px) !important;
    border-color: rgba(2, 132, 199, 0.4) !important;
    box-shadow: 0 12px 28px -5px rgba(15, 23, 42, 0.08) !important;
}

.sg-card-header-light {
    font-size: 14px !important;
    font-weight: 700 !important;
    color: #0f172a !important;
    margin-bottom: 12px !important;
    display: flex !important;
    align-items: center !important;
    gap: 8px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.05em !important;
}

/* Glowing Primary Button */
.sg-btn-light {
    background: linear-gradient(135deg, #0284c7 0%, #2563eb 50%, #7c3aed 100%) !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    border: none !important;
    border-radius: 14px !important;
    padding: 13px 26px !important;
    box-shadow: 0 4px 14px rgba(2, 132, 199, 0.35) !important;
    cursor: pointer !important;
    transition: all 0.25s ease !important;
    width: 100% !important;
}

.sg-btn-light:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(2, 132, 199, 0.5) !important;
    filter: brightness(1.05) !important;
}

.sg-btn-light:active {
    transform: translateY(0) !important;
}

/* Light Component Form Elements */
.gradio-container input,
.gradio-container textarea,
.gradio-container .block,
.gradio-container .form,
.gradio-container fieldset,
.gradio-container .input-container {
    background-color: #ffffff !important;
    color: #0f172a !important;
    border-color: rgba(203, 213, 225, 0.8) !important;
    border-radius: 12px !important;
}

.gradio-container input:focus,
.gradio-container textarea:focus {
    border-color: #0284c7 !important;
    box-shadow: 0 0 0 3px rgba(2, 132, 199, 0.15) !important;
}

.gradio-container label,
.gradio-container label span {
    color: #334155 !important;
    font-weight: 600 !important;
}

/* Grid & Metadata Rows */
.sg-info-grid-light {
    display: grid !important;
    grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)) !important;
    gap: 16px !important;
    margin-top: 18px !important;
}

.sg-meta-row-light {
    display: flex !important;
    justify-content: space-between !important;
    align-items: center !important;
    font-size: 13px !important;
    padding: 7px 0 !important;
    border-bottom: 1px dashed rgba(203, 213, 225, 0.7) !important;
}

.sg-meta-row-light:last-child {
    border-bottom: none !important;
}

.sg-meta-row-light span {
    color: #64748b !important;
}

.sg-meta-row-light strong {
    color: #0284c7 !important;
    font-weight: 600 !important;
}

.sg-unit-box-light {
    background: rgba(16, 185, 129, 0.08) !important;
    border: 1px solid rgba(16, 185, 129, 0.3) !important;
    border-radius: 12px !important;
    padding: 12px 14px !important;
    margin: 10px 0 !important;
    text-align: center !important;
}

.sg-unit-val-light {
    font-size: 19px !important;
    font-weight: 800 !important;
    color: #059669 !important;
    letter-spacing: -0.01em !important;
}

.sg-unit-desc-light {
    font-size: 11px !important;
    color: #047857 !important;
    margin-top: 3px !important;
}

/* Footer */
.sg-footer-light {
    text-align: center !important;
    padding: 24px 16px 12px !important;
    margin-top: 28px !important;
    border-top: 1px solid rgba(226, 232, 240, 0.8) !important;
    color: #64748b !important;
    font-size: 12px !important;
    line-height: 1.6 !important;
}

.sg-footer-title-light {
    font-weight: 700 !important;
    color: #1e293b !important;
    font-size: 13px !important;
    margin-bottom: 4px !important;
}
"""

with gr.Blocks(title="Indian Electricity Load Forecasting - Hybrid Ensemble") as demo:
    gr.HTML(f"<style>{LIGHT_THEME_CSS}</style>")
    
    # Modern Light-Themed Glassmorphism Header
    gr.HTML("""
    <div class="sg-header-light">
        <div class="sg-badge-light">⚡ SMART GRID TELEMETRY • WEIGHTED HYBRID ENSEMBLE</div>
        <div class="sg-title-light">🇮🇳 Indian Smart Grid</div>
        <div class="sg-title-sub-light">National Electricity Demand &amp; Load Forecasting</div>
        <div class="sg-model-tag-light">⚡ Weighted Hybrid Ensemble Engine (LR + SVR + RF + XGB + LSTM)</div>
        <p class="sg-subtitle-light">AI-powered multi-model telemetry forecasting of India's national electricity demand</p>
    </div>
    """)
    
    # Input & Output Panels
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            gr.HTML("""
            <div class="sg-card-header-light">
                <span>🔮</span>
                <strong>Prediction Input Parameters</strong>
            </div>
            """)
            with gr.Row():
                date_input = gr.Textbox(
                    label="📅 Date (YYYY-MM-DD)",
                    placeholder="e.g. 2024-04-01",
                    value="2024-04-01",
                    info="Date format: YYYY-MM-DD"
                )
                time_input = gr.Textbox(
                    label="🕐 Time (24-Hour Format)",
                    placeholder="e.g. 18:00",
                    value="18:00",
                    info="Time format: 24-hour (HH:00)"
                )
            predict_button = gr.Button(
                "⚡ Predict Electricity Demand",
                variant="primary",
                elem_classes=["sg-btn-light"]
            )
            
        with gr.Column(scale=1):
            gr.HTML("""
            <div class="sg-card-header-light">
                <span>📊</span>
                <strong>National Grid Telemetry Result</strong>
            </div>
            """)
            output = gr.Textbox(
                label="🇮🇳 Predicted National Electricity Demand (MW)",
                placeholder="Prediction result with MW unit will appear here...",
                interactive=False,
                lines=8
            )
            
    # Glass Information Cards Grid
    gr.HTML("""
    <div class="sg-info-grid-light">
        <div class="sg-card-light">
            <div class="sg-card-header-light">⚡ Electricity Demand Unit</div>
            <div class="sg-unit-box-light">
                <div class="sg-unit-val-light">MW — Megawatts</div>
                <div class="sg-unit-desc-light">Standard unit of national grid electrical power</div>
            </div>
            <p style="font-size: 12px; color: #64748b; margin: 6px 0 0 0; line-height: 1.5;">
                Electricity demand represents the instantaneous active electrical power required across the Indian National Grid.
            </p>
        </div>
        <div class="sg-card-light">
            <div class="sg-card-header-light">🤖 Hybrid Architecture Weights</div>
            <div class="sg-meta-row-light"><span>Linear Regression:</span> <strong>5% (0.05)</strong></div>
            <div class="sg-meta-row-light"><span>SVR (RBF Kernel):</span> <strong>15% (0.15)</strong></div>
            <div class="sg-meta-row-light"><span>Random Forest:</span> <strong>25% (0.25)</strong></div>
            <div class="sg-meta-row-light"><span>XGBoost Regressor:</span> <strong>30% (0.30)</strong></div>
            <div class="sg-meta-row-light"><span>LSTM Deep Learning:</span> <strong>25% (0.25)</strong></div>
            <div class="sg-meta-row-light"><span>Total Ensemble Weight:</span> <strong>100% (1.00)</strong></div>
        </div>
        <div class="sg-card-light">
            <div class="sg-card-header-light">🇮🇳 Dataset Telemetry</div>
            <div class="sg-meta-row-light"><span>Source:</span> <strong>Indian Hourly Electricity Demand</strong></div>
            <div class="sg-meta-row-light"><span>Target:</span> <strong>National Hourly Demand</strong></div>
            <div class="sg-meta-row-light"><span>Frequency:</span> <strong>Hourly (24 Timesteps / Day)</strong></div>
            <div class="sg-meta-row-light"><span>Lookback Window:</span> <strong>24 Hours (LSTM)</strong></div>
            <div class="sg-meta-row-light"><span>Evaluation Metric:</span> <strong>MAE, MSE, RMSE, R²</strong></div>
        </div>
    </div>
    <div class="sg-footer-light">
        <div class="sg-footer-title-light">🇮🇳 Indian Smart Grid Load Forecasting System</div>
        <div>Machine Learning &amp; Deep Learning Project • LR (5%) | SVR (15%) | RF (25%) | XGBoost (30%) | LSTM (25%)</div>
        <div style="margin-top: 4px; color: #94a3b8;">Academic PBL Project • 5th Semester</div>
    </div>
    """)

    # Connect Prediction Function
    predict_button.click(
        fn=predict_hybrid_load,
        inputs=[date_input, time_input],
        outputs=output
    )

# Launch Gradio Demo
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
